In [1]:
# Air Traffic Data Analysis using PySpark

## Project Objective
#This project demonstrates core Spark DataFrame operations using PySpark on air traffic data.  
#It recreates important Spark concepts such as schema inspection, selection, filtering, sorting, sampling, distinct counts, type casting, and repartitioning in a local environment.

## Tools Used
#- Python
#- PySpark
#- Jupyter Notebook
#- Spark DataFrames

In [2]:
import os
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.driver.host=127.0.0.1 --conf spark.driver.bindAddress=127.0.0.1 pyspark-shell"

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, column
from pyspark.sql.types import StructField, StructType, StringType, LongType

In [4]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Air Traffic Data Analysis using PySpark") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 13:54:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
print(spark)
print(spark.version)

4.1.1


In [6]:
import os
print(os.getcwd())
print(os.listdir(".."))

/Users/mainak
['mainak', '.localized', 'Shared']


In [7]:
df_csv = spark.read.csv("/Users/mainak/Documents/Air Traffic Data Analysis using PySpark/2015-summary.csv", header=True, inferSchema=True)
df_csv.show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
+-----------------+-------------------+-----+
only showing top 5 rows


In [8]:
df_csv.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: integer (nullable = true)



In [9]:
myManualSchema=StructType([
    StructField("DEST_COUNTRY_NAME",StringType(),True),
    StructField("ORIGIN_COUNTRY_NAME",StringType(),True),
    StructField("count",LongType(),True),
])

In [10]:
df = spark.read.csv("/Users/mainak/Documents/Air Traffic Data Analysis using PySpark/2015-summary.csv", header=True, schema=myManualSchema)

In [11]:
df.columns

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']

In [12]:
df.select("DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME").show(10)

+-----------------+-------------------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|
+-----------------+-------------------+
|    United States|            Romania|
|    United States|            Croatia|
|    United States|            Ireland|
|            Egypt|      United States|
|    United States|              India|
|    United States|          Singapore|
|    United States|            Grenada|
|       Costa Rica|      United States|
|          Senegal|      United States|
|          Moldova|      United States|
+-----------------+-------------------+
only showing top 10 rows


In [13]:
df.select(
    expr("DEST_COUNTRY_NAME"),
    col("ORIGIN_COUNTRY_NAME"),
    column("count")
).show(7)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|   15|
|    United States|            Croatia|    1|
|    United States|            Ireland|  344|
|            Egypt|      United States|   15|
|    United States|              India|   62|
|    United States|          Singapore|    1|
|    United States|            Grenada|   62|
+-----------------+-------------------+-----+
only showing top 7 rows


In [14]:
df_with_domestic = df.selectExpr(
    "DEST_COUNTRY_NAME",
    "ORIGIN_COUNTRY_NAME",
    "count",
    "(DEST_COUNTRY_NAME = ORIGIN_COUNTRY_NAME) as Domestic"
)

df_with_domestic.show(9)

+-----------------+-------------------+-----+--------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|Domestic|
+-----------------+-------------------+-----+--------+
|    United States|            Romania|   15|   false|
|    United States|            Croatia|    1|   false|
|    United States|            Ireland|  344|   false|
|            Egypt|      United States|   15|   false|
|    United States|              India|   62|   false|
|    United States|          Singapore|    1|   false|
|    United States|            Grenada|   62|   false|
|       Costa Rica|      United States|  588|   false|
|          Senegal|      United States|   40|   false|
+-----------------+-------------------+-----+--------+
only showing top 9 rows


In [15]:
df.selectExpr(
    "count(distinct(ORIGIN_COUNTRY_NAME)) as dintinct_origin_countries",
    "count(distinct(DEST_COUNTRY_NAME)) as distinct_destination_countries").show()

+-------------------------+------------------------------+
|dintinct_origin_countries|distinct_destination_countries|
+-------------------------+------------------------------+
|                      125|                           132|
+-------------------------+------------------------------+



In [16]:
df_with_domestic.drop("Domestic").columns

['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count']

In [17]:
df_numcount = df.withColumn("numcount", col("count").cast("long"))
df_numcount.printSchema()

root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)
 |-- numcount: long (nullable = true)



In [18]:
#Filter flights not from singapore and count less than 3
df.filter(col("count")<3).where(col("ORIGIN_COUNTRY_NAME")!="Singapore").show(10)



+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Croatia|    1|
|             Moldova|      United States|    1|
|               Malta|      United States|    1|
|       United States|          Gibraltar|    1|
|Saint Vincent and...|      United States|    1|
|            Suriname|      United States|    1|
|       United States|             Cyprus|    1|
|             Liberia|      United States|    2|
|             Hungary|      United States|    2|
|       United States|            Vietnam|    2|
+--------------------+-------------------+-----+
only showing top 10 rows


In [19]:
#Count distinct destinations from United States
distinct_count= df.filter(col("ORIGIN_COUNTRY_NAME")=="United States").select("DEST_COUNTRY_NAME").distinct().count()
print("Distinct destinations from United States:", distinct_count)

Distinct destinations from United States: 132


In [20]:
#Random sample of 70%
seed = 5
withReplacement = False
fraction = 0.7

sample_count = df.sample(withReplacement, fraction, seed).count()
print("Sampled row count:", sample_count)

Sampled row count: 189


In [21]:
#Top 10 destinations sorted by count descending and destination ascending to show which country has the highest count
df.orderBy(col("count").desc(),col("DEST_COUNTRY_NAME").asc()).limit(10).show(10)


+-----------------+-------------------+------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME| count|
+-----------------+-------------------+------+
|    United States|      United States|370002|
|    United States|             Canada|  8483|
|           Canada|      United States|  8399|
|    United States|             Mexico|  7187|
|           Mexico|      United States|  7140|
|   United Kingdom|      United States|  2025|
|    United States|     United Kingdom|  1970|
|            Japan|      United States|  1548|
|    United States|              Japan|  1496|
|          Germany|      United States|  1468|
+-----------------+-------------------+------+



In [22]:
df_repartitioned = df.repartition(col("ORIGIN_COUNTRY_NAME"))
print("Repartitioning completed.")

Repartitioning completed.


In [23]:
origin_summary = df.groupBy("ORIGIN_COUNTRY_NAME").sum("count").withColumnRenamed("sum(count)", "total_flights").orderBy(col("total_flights").desc())

origin_summary.show(10)

+-------------------+-------------+
|ORIGIN_COUNTRY_NAME|total_flights|
+-------------------+-------------+
|      United States|       411966|
|             Canada|         8483|
|             Mexico|         7187|
|     United Kingdom|         1970|
|              Japan|         1496|
| Dominican Republic|         1420|
|            Germany|         1336|
|        The Bahamas|          986|
|             France|          952|
|              China|          920|
+-------------------+-------------+
only showing top 10 rows
